# Linear Regression - Ridge tuning

### __Introduction__

- We will start by using this model mostly as a baseline for our other models.
- A priori, we think that this will most likely not be our best performing model, mostly because it assumes linearity of features, which may not be the case. 

## 1. 

In [1]:
# 1) Imports 
import numpy as np
import pandas as pd

from datetime import datetime

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.model_selection import ParameterSampler

# 2) Import pre processing helpers
#    -> this should define: full_train_dataset, cat_feat, num_feat,
#       basic_string_transformer, def_string_basic_transformer,
#       preprocess_categorical, MyTargetEncoder, MyOneHotEncoder, etc.
%run 05_0_preproc_helpers.ipynb  

# 3) Define target
TARGET_COL = "price"

# 4) Separate X and y from the treated dataset
y = full_train_dataset[TARGET_COL].copy()
X = full_train_dataset.drop(columns=[TARGET_COL]).copy()

# Numerical and categorical features (splitted in pre processing)
numeric_features = num_feat
categorical_features = cat_feat

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Num features:", numeric_features)
print("Cat features:", categorical_features)

X shape: (75973, 10)
y shape: (75973,)
Num features: ['year', 'mileage', 'engineSize', 'tax', 'mpg', 'previousOwners']
Cat features: ['Brand', 'model', 'transmission', 'fuelType']


In [2]:
valid_transmissions = ["MANUAL", "AUTOMATIC", "SEMIAUTO"]
valid_fueltypes    = ["PETROL", "DIESEL", "HYBRID"]

# KFold config
N_SPLITS = 8
RANDOM_STATE = 42

kf = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

# Hyperparameter space for Ridge
ridge_param_distributions = {
    "alpha": np.logspace(-3, 3, 40)  # 1e-3 to 1e3
}

N_RANDOM_CONFIGS_RIDGE = 40

ridge_sampler = ParameterSampler(
    ridge_param_distributions,
    n_iter=N_RANDOM_CONFIGS_RIDGE,
    random_state=RANDOM_STATE
)

ridge_search_results = []

# Best configs
ridge_best_rmse = np.inf
ridge_best_config_rmse = None

ridge_best_mae = np.inf
ridge_best_config_mae = None

ridge_best_combo = np.inf
ridge_best_config_combo = None

ridge_log_path = "ridge_random_search_log.txt"

with open(ridge_log_path, "w", encoding="utf-8") as log_file:

    def log_ridge(msg: str):
        log_file.write(datetime.now().strftime("[%Y-%m-%d %H:%M:%S] ") + msg + "\n")
        log_file.flush()

    log_ridge("# =============================")
    log_ridge("# START OF RANDOM SEARCH Ridge")
    log_ridge("# =============================")
    log_ridge(f"N_SPLITS = {N_SPLITS}, N_RANDOM_CONFIGS = {N_RANDOM_CONFIGS_RIDGE}")
    log_ridge(f"param_distributions = {ridge_param_distributions}")

    for config_id, params in enumerate(ridge_sampler, start=1):
        log_ridge("")
        log_ridge(f"######## CONFIG {config_id}/{N_RANDOM_CONFIGS_RIDGE} ########")
        log_ridge(f"Parameters: {params}")

        fold_rmses_val = []
        fold_maes_val  = []
        fold_r2s_val   = []
        fold_bias_val  = []

        fold_rmses_tr = []
        fold_maes_tr  = []
        fold_r2s_tr   = []
        fold_bias_tr  = []

        for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
            log_ridge("")
            log_ridge(f"[CONFIG {config_id}] ==== FOLD {fold}/{N_SPLITS} ====")

            # 1) split
            X_train = X.iloc[train_idx].copy()
            X_val   = X.iloc[val_idx].copy()
            y_train = y.iloc[train_idx].copy()
            y_val   = y.iloc[val_idx].copy()

            log_ridge(f"[C{config_id}|F{fold}] X_train shape: {X_train.shape}, X_val shape: {X_val.shape}")
            log_ridge(f"[C{config_id}|F{fold}] y_train shape: {y_train.shape}, y_val shape: {y_val.shape}")

            # NaNs before numeric imputation
            log_ridge(f"[C{config_id}|F{fold}] NaNs before numeric imputation (num features):")
            log_ridge(str(X_train[numeric_features].isna().sum()))

            log_ridge(f"[C{config_id}|F{fold}] NaNs before (categoricals):")
            log_ridge(str(X_train[categorical_features].isna().sum()))

            unknown_counts_before = (X_train[categorical_features] == "UNKNOWN").sum()
            log_ridge(f"[C{config_id}|F{fold}] 'UNKNOWN' before (categoricals):")
            log_ridge(str(unknown_counts_before))

            # 2) NUMERIC PREPROCESSING
            year_state = fit_year_median(
                X_train,
                year_col="year",
                model_col="model"
            )
            X_train = transform_year_with_model_median(X_train, state=year_state)
            X_val   = transform_year_with_model_median(X_val,   state=year_state)

            mileage_state = fit_mileage_imputer(
                X_train,
                mileage_col="mileage",
                do_abs=True
            )
            X_train = transform_mileage_imputer(X_train, state=mileage_state)
            X_val   = transform_mileage_imputer(X_val,   state=mileage_state)

            engine_state = fit_engine_size_imputer(
                X_train,
                engine_col="engineSize"
            )
            X_train = transform_engine_size_imputer(X_train, state=engine_state)
            X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

            tax_state = fit_tax_imputer(
                X_train,
                tax_col="tax",
                do_abs=True
            )
            X_train = transform_tax_imputer(X_train, state=tax_state)
            X_val   = transform_tax_imputer(X_val,   state=tax_state)

            mpg_state = fit_mpg_imputer(
                X_train,
                mpg_col="mpg",
                do_abs=True
            )
            X_train = transform_mpg_imputer(X_train, state=mpg_state)
            X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

            owners_state = fit_previous_owners_imputer(
                X_train,
                owners_col="previousOwners",
                year_col="year",
                mileage_col="mileage"
            )
            X_train = transform_previous_owners_imputer(X_train, state=owners_state)
            X_val   = transform_previous_owners_imputer(X_val,   state=owners_state)

            log_ridge(f"[C{config_id}|F{fold}] After numeric imputation: X_train shape = {X_train.shape}, X_val shape = {X_val.shape}")
            log_ridge(f"[C{config_id}|F{fold}] NaNs after numeric imputation (num features):")
            log_ridge(str(X_train[numeric_features].isna().sum()))

            # 3) CATEGORICAL RESOLVERS
            brand_state = fit_ambiguous_brand_resolver(
                train_df=X_train,
                valid_brands=valid_brands,
                brand_col="Brand",
                model_col="model",
                year_col="year",
            )
            X_train, _, brand_still_invalid_train = transform_ambiguous_brands(X_train, brand_state)
            X_val,   _, brand_still_invalid_val   = transform_ambiguous_brands(X_val,   brand_state)
            log_ridge(f"[C{config_id}|F{fold}] After solving Brand - still invalid (train): {len(brand_still_invalid_train)}")

            model_state = fit_invalid_model_resolver(
                train_df=X_train,
                valid_models_by_brand=valid_models_by_brand,
                brand_col="Brand",
                model_col="model",
                year_col="year",
                fuel_col="fuelType",
                mpg_col="mpg",
            )
            X_train, _, model_still_invalid_train = transform_invalid_models(X_train, model_state)
            X_val,   _, model_still_invalid_val   = transform_invalid_models(X_val,   model_state)
            log_ridge(f"[C{config_id}|F{fold}] After solving model - still invalid (train): {len(model_still_invalid_train)}")

            transm_state = fit_transmission_resolver(
                train_df=X_train,
                valid_transmissions=valid_transmissions,
                transm_col="transmission",
                brand_col="Brand",
                model_col="model",
                fuel_col="fuelType",
            )
            X_train, _, transm_still_invalid_train = transform_transmission_resolver(X_train, transm_state)
            X_val,   _, transm_still_invalid_val   = transform_transmission_resolver(X_val,   transm_state)
            log_ridge(f"[C{config_id}|F{fold}] After solving transmission - distinct (train): " +
                      str(sorted(X_train["transmission"].astype(str).str.upper().unique())))

            fuel_state = fit_fueltype_resolver(
                train_df=X_train,
                valid_fueltypes=valid_fueltypes,
                fuel_col="fuelType",
                brand_col="Brand",
                model_col="model",
                transm_col="transmission",
            )
            X_train, _, fuel_still_invalid_train = transform_fueltype_resolver(X_train, fuel_state)
            X_val,   _, fuel_still_invalid_val   = transform_fueltype_resolver(X_val,   fuel_state)
            log_ridge(f"[C{config_id}|F{fold}] After solving fuelType - distinct (train): " +
                      str(sorted(X_train["fuelType"].astype(str).str.upper().unique())))

            # 4) CATEGORICAL ENCODING
            high_card_features = ["Brand", "model"]
            low_card_features  = [c for c in categorical_features if c not in high_card_features]

            log_ridge(f"[C{config_id}|F{fold}] high_card_features = {high_card_features}")
            log_ridge(f"[C{config_id}|F{fold}] low_card_features  = {low_card_features}")

            te = MyTargetEncoder(smoothing=5)
            te.fit(X_train[high_card_features], y_train)
            X_train_high = te.transform(X_train[high_card_features])
            X_val_high   = te.transform(X_val[high_card_features])

            ohe = MyOneHotEncoder()
            ohe.fit(X_train[low_card_features])
            X_train_low = ohe.transform(X_train[low_card_features])
            X_val_low   = ohe.transform(X_val[low_card_features])

            X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
            X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

            log_ridge(f"[C{config_id}|F{fold}] X_train_cat shape: {X_train_cat.shape}")
            log_ridge(f"[C{config_id}|F{fold}] X_val_cat   shape: {X_val_cat.shape}")

            # 5) NUMERIC SCALING
            scaler = StandardScaler()
            X_train_num = scaler.fit_transform(X_train[numeric_features])
            X_val_num   = scaler.transform(X_val[numeric_features])

            X_train_num_df = pd.DataFrame(
                X_train_num,
                index=X_train.index,
                columns=[f"num_{col}" for col in numeric_features]
            )
            X_val_num_df = pd.DataFrame(
                X_val_num,
                index=X_val.index,
                columns=[f"num_{col}" for col in numeric_features]
            )

            log_ridge(f"[C{config_id}|F{fold}] X_train_num_df shape: {X_train_num_df.shape}")
            log_ridge(f"[C{config_id}|F{fold}] X_val_num_df   shape: {X_val_num_df.shape}")

            # 6) FINAL MATRIX
            X_train_final = pd.concat([X_train_num_df, X_train_cat], axis=1)
            X_val_final   = pd.concat([X_val_num_df,   X_val_cat],   axis=1)

            log_ridge(f"[C{config_id}|F{fold}] X_train_final shape: {X_train_final.shape}")
            log_ridge(f"[C{config_id}|F{fold}] X_val_final   shape: {X_val_final.shape}")

            # 7) RIDGE MODEL
            ridge = Ridge(
                alpha=params["alpha"],
                fit_intercept=True
            )

            log_ridge(f"[C{config_id}|F{fold}] Training Ridge...")
            ridge.fit(X_train_final, y_train)

            y_pred_train = ridge.predict(X_train_final)
            y_pred_val   = ridge.predict(X_val_final)

            # 8) METRICS
            mse_val  = mean_squared_error(y_val, y_pred_val)
            rmse_val = np.sqrt(mse_val)
            mae_val  = mean_absolute_error(y_val, y_pred_val)
            r2_val   = r2_score(y_val, y_pred_val)
            bias_val = float(np.mean(y_pred_val - y_val))

            mse_tr  = mean_squared_error(y_train, y_pred_train)
            rmse_tr = np.sqrt(mse_tr)
            mae_tr  = mean_absolute_error(y_train, y_pred_train)
            r2_tr   = r2_score(y_train, y_pred_train)
            bias_tr = float(np.mean(y_pred_train - y_train))

            log_ridge(f"[C{config_id}|F{fold}] TRAIN -> RMSE: {rmse_tr:.2f} | MAE: {mae_tr:.2f} | R2: {r2_tr:.4f} | Bias(pred-true): {bias_tr:.2f}")
            log_ridge(f"[C{config_id}|F{fold}] VAL   -> RMSE: {rmse_val:.2f} | MAE: {mae_val:.2f} | R2: {r2_val:.4f} | Bias(pred-true): {bias_val:.2f}")

            fold_rmses_tr.append(rmse_tr)
            fold_maes_tr.append(mae_tr)
            fold_r2s_tr.append(r2_tr)
            fold_bias_tr.append(bias_tr)

            fold_rmses_val.append(rmse_val)
            fold_maes_val.append(mae_val)
            fold_r2s_val.append(r2_val)
            fold_bias_val.append(bias_val)

        # mean over folds
        mean_rmse_val = np.mean(fold_rmses_val)
        mean_mae_val  = np.mean(fold_maes_val)
        mean_r2_val   = np.mean(fold_r2s_val)
        mean_bias_val = np.mean(fold_bias_val)

        mean_rmse_tr = np.mean(fold_rmses_tr)
        mean_mae_tr  = np.mean(fold_maes_tr)
        mean_r2_tr   = np.mean(fold_r2s_tr)
        mean_bias_tr = np.mean(fold_bias_tr)

        combo_score = 0.5 * mean_rmse_val + 0.5 * mean_mae_val

        log_ridge("")
        log_ridge(f"Config {config_id} - Avg TRAIN RMSE: {mean_rmse_tr:.2f} | MAE: {mean_mae_tr:.2f} | R2: {mean_r2_tr:.4f} | Bias: {mean_bias_tr:.2f}")
        log_ridge(f"Config {config_id} - Avg VAL   RMSE: {mean_rmse_val:.2f} | MAE: {mean_mae_val:.2f} | R2: {mean_r2_val:.4f} | Bias: {mean_bias_val:.2f}")
        log_ridge(f"Config {config_id} - Combined score (0.5*RMSE + 0.5*MAE) [VAL]: {combo_score:.2f}")

        ridge_search_results.append({
            "config_id": config_id,
            **params,
            "rmse_train_mean": mean_rmse_tr,
            "mae_train_mean": mean_mae_tr,
            "r2_train_mean": mean_r2_tr,
            "bias_train_mean": mean_bias_tr,
            "rmse_mean": mean_rmse_val,
            "mae_mean": mean_mae_val,
            "r2_mean": mean_r2_val,
            "bias_mean": mean_bias_val,
            "combo_score": combo_score,
        })

        # best by RMSE
        if mean_rmse_val < ridge_best_rmse:
            ridge_best_rmse = mean_rmse_val
            ridge_best_config_rmse = {**params}
            log_ridge(f"[NEW BEST RMSE] Config {config_id} with avg RMSE (VAL) = {ridge_best_rmse:.2f}")

        # best by MAE
        if mean_mae_val < ridge_best_mae:
            ridge_best_mae = mean_mae_val
            ridge_best_config_mae = {**params}
            log_ridge(f"[NEW BEST MAE] Config {config_id} with avg MAE (VAL) = {ridge_best_mae:.2f}")

        # best by combo
        if combo_score < ridge_best_combo:
            ridge_best_combo = combo_score
            ridge_best_config_combo = {**params}
            log_ridge(f"[NEW BEST COMBINED] Config {config_id} with score = {ridge_best_combo:.2f}")

    log_ridge("")
    log_ridge("# =============================")
    log_ridge("# END OF RANDOM SEARCH Ridge")
    log_ridge("# =============================")
    log_ridge(f"Best configuration (min RMSE VAL): {ridge_best_config_rmse}")
    log_ridge(f"Best average RMSE (VAL): {ridge_best_rmse:.2f}")
    log_ridge(f"Best configuration (min MAE VAL): {ridge_best_config_mae}")
    log_ridge(f"Best average MAE  (VAL): {ridge_best_mae:.2f}")
    log_ridge(f"Best configuration (combined score VAL): {ridge_best_config_combo}")
    log_ridge(f"Best combined score (VAL): {ridge_best_combo:.2f}")

ridge_results_df = pd.DataFrame(ridge_search_results)
ridge_results_df_sorted = ridge_results_df.sort_values(by="mae_mean", ascending=True)

display(ridge_results_df_sorted.head(10))

print("\nBest configuration found (min RMSE VAL):")
print(ridge_best_config_rmse)
print("Best average RMSE (VAL):", ridge_best_rmse)

print("\nBest configuration found (min MAE VAL):")
print(ridge_best_config_mae)
print("Best average MAE (VAL):", ridge_best_mae)

print("\nBest configuration found (VAL = 0.5*RMSE + 0.5*MAE):")
print(ridge_best_config_combo)
print("Best combined score (VAL):", ridge_best_combo)

ridge_results_df_sorted.to_csv("ridge_random_search_results.csv", index=False)
print(f"\nDetailed logs in: {ridge_log_path}")


,config_id,alpha,rmse_train_mean,mae_train_mean,r2_train_mean,bias_train_mean,rmse_mean,mae_mean,r2_mean,bias_mean,combo_score
39,40,1000.000000,4160.877907,2639.831550,0.817384,-6.596358e-13,4186.211135,2645.944558,0.815094,0.122279,3416.077846
38,39,701.703829,4159.855033,2641.573172,0.817474,7.189970e-13,4185.069911,2647.679375,0.815195,0.121792,3416.374643
37,38,492.388263,4159.241720,2643.005068,0.817528,1.796204e-13,4184.375204,2649.092823,0.815257,0.121328,3416.734014
36,37,345.510729,4158.871864,2644.137583,0.817560,6.681657e-14,4183.950981,2650.213324,0.815294,0.120642,3417.082152
35,36,242.446202,4158.645727,2644.999705,0.817580,7.847909e-13,4183.689862,2651.075069,0.815318,0.119666,3417.382465
34,35,170.125428,4158.505274,2645.647747,0.817593,-2.292038e-13,4183.527955,2651.716121,0.815332,0.118461,3417.622038
33,34,119.377664,4158.417430,2646.133440,0.817600,5.982125e-13,4183.427770,2652.191090,0.815341,0.117147,3417.809430
32,33,83.767764,4158.362951,2646.494668,0.817605,-5.680659e-13,4183.366846,2652.550153,0.815347,0.115847,3417.958499
31,32,58.780161,4158.329892,2646.761608,0.817608,8.669454e-13,4183.331035,2652.817044,0.815350,0.114647,3418.074039
30,31,41.246264,4158.310366,2646.956183,0.817610,2.833533e-15,4183.311049,2653.014846,0.815352,0.113588,3418.162947



Best configuration found (min RMSE VAL):
{'alpha': np.float64(14.251026703029993)}
Best average RMSE (VAL): 4183.29550599769

Best configuration found (min MAE VAL):
{'alpha': np.float64(1000.0)}
Best average MAE (VAL): 2645.944557659762

Best configuration found (VAL = 0.5*RMSE + 0.5*MAE):
{'alpha': np.float64(1000.0)}
Best combined score (VAL): 3416.077846189613

Detailed logs in: ridge_random_search_log.txt


In [3]:
# Choose final hyperparameters from Ridge search
final_config_ridge = ridge_best_config_rmse
#final_config_ridge = {
#    "alpha": ridge_best_config_rmse["alpha"]  
#}
print("Final Ridge config used for train:", final_config_ridge)

print("Preparing final Ridge Regression training...")

# 1) PREPARE FULL TRAIN FEATURES
X_full = X.copy()
y_full = y.copy()

# a) STRING NORMALIZATION 
cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]

for col in cols_to_normalize:
    if col in X_full.columns:
        X_full[col] = X_full[col].apply(
            lambda x: basic_string_transformer(
                x, 
                remove_middle_spaces=True, 
                allow_extra_chars=""
            )
        )

# Define feature sets
high_card_features = ["Brand", "model"] 
low_card_features  = [c for c in categorical_features if c not in high_card_features]

# 2) NUMERIC PRE PROCESSING - FULL TRAIN (FIT & TRANSFORM)
# Year
year_state = fit_year_median(X_full, year_col="year", model_col="model")
X_full = transform_year_with_model_median(X_full, state=year_state)

# Mileage
mileage_state = fit_mileage_imputer(X_full, mileage_col="mileage", do_abs=True)
X_full = transform_mileage_imputer(X_full, state=mileage_state)

# Engine Size
engine_state = fit_engine_size_imputer(X_full, engine_col="engineSize")
X_full = transform_engine_size_imputer(X_full, state=engine_state)

# Tax
tax_state = fit_tax_imputer(X_full, tax_col="tax", do_abs=True)
X_full = transform_tax_imputer(X_full, state=tax_state)

# MPG
mpg_state = fit_mpg_imputer(X_full, mpg_col="mpg", do_abs=True)
X_full = transform_mpg_imputer(X_full, state=mpg_state)

# Paint Quality 
# paint_state = fit_paint_quality_imputer(X_full, paint_col="paintQuality%")
# X_full = transform_paint_quality_imputer(X_full, state=paint_state)

# Previous Owners
owners_state = fit_previous_owners_imputer(
    X_full, owners_col="previousOwners", year_col="year", mileage_col="mileage"
)
X_full = transform_previous_owners_imputer(X_full, state=owners_state)

# 3) CATEGORICAL RESOLVERS - FULL TRAIN (FIT & TRANSFORM)
# Brand
brand_state = fit_ambiguous_brand_resolver(
    train_df=X_full, valid_brands=valid_brands, 
    brand_col="Brand", model_col="model", year_col="year"
)
X_full, _, brand_still_invalid_full = transform_ambiguous_brands(X_full, brand_state)
print(f"Train full - Invalid Brands remaining: {len(brand_still_invalid_full)}")

# Model
model_state = fit_invalid_model_resolver(
    train_df=X_full, valid_models_by_brand=valid_models_by_brand,
    brand_col="Brand", model_col="model", year_col="year", 
    fuel_col="fuelType", mpg_col="mpg"
)
X_full, _, model_still_invalid_full = transform_invalid_models(X_full, model_state)
print(f"Train full - Invalid Models remaining: {len(model_still_invalid_full)}")

# Transmission
transm_state = fit_transmission_resolver(
    train_df=X_full, valid_transmissions=valid_transmissions,
    transm_col="transmission", brand_col="Brand", 
    model_col="model", fuel_col="fuelType"
)
X_full, _, _ = transform_transmission_resolver(X_full, transm_state)

# Fuel Type
fuel_state = fit_fueltype_resolver(
    train_df=X_full, valid_fueltypes=valid_fueltypes,
    fuel_col="fuelType", brand_col="Brand", 
    model_col="model", transm_col="transmission"
)
X_full, _, _ = transform_fueltype_resolver(X_full, fuel_state)

# 4) CATEGORICAL ENCODING - FULL TRAIN (FIT & TRANSFORM)
# Target Encoding
te = MyTargetEncoder(smoothing=5)
te.fit(X_full[high_card_features], y_full)
X_full_high = te.transform(X_full[high_card_features])

# One-Hot Encoding
ohe = MyOneHotEncoder()
ohe.fit(X_full[low_card_features])
X_full_low = ohe.transform(X_full[low_card_features])

X_full_cat = pd.concat([X_full_high, X_full_low], axis=1)
print("X_full_cat shape:", X_full_cat.shape)

# 5) NUMERIC SCALING - FULL TRAIN (FIT & TRANSFORM)
scaler = StandardScaler()
X_full_num = scaler.fit_transform(X_full[numeric_features])

X_full_num_df = pd.DataFrame(
    X_full_num,
    index=X_full.index,
    columns=[f"num_{col}" for col in numeric_features]
)
print("X_full_num_df shape:", X_full_num_df.shape)

# 6) FINAL MATRIX - FULL TRAIN
X_full_final = pd.concat([X_full_num_df, X_full_cat], axis=1)
print("X_full_final shape:", X_full_final.shape)

# 7) TRAIN FINAL RIDGE REGRESSION MODEL
ridge_final = Ridge(
    alpha=final_config_ridge["alpha"],
    fit_intercept=True
)

print("Training final Ridge Regression model on full data...")
ridge_final.fit(X_full_final, y_full)
print("Done.")

# 8) PREPARE TEST FEATURES
test_df = pd.read_csv("../../project_data/test.csv")

# a) STRING NORMALIZATION (same logic as train)
for col in cols_to_normalize:
    if col in test_df.columns:
        test_df[col] = test_df[col].apply(
            lambda x: basic_string_transformer(
                x, 
                remove_middle_spaces=True, 
                allow_extra_chars=""
            )
        )

X_test = test_df[numeric_features + categorical_features].copy()

# b) NUMERIC PREPROCESSING - TEST (TRANSFORM ONLY)
X_test = transform_year_with_model_median(X_test, state=year_state)
X_test = transform_mileage_imputer(X_test, state=mileage_state)
X_test = transform_engine_size_imputer(X_test, state=engine_state)
X_test = transform_tax_imputer(X_test, state=tax_state)
X_test = transform_mpg_imputer(X_test, state=mpg_state)
# X_test = transform_paint_quality_imputer(X_test, state=paint_state)
X_test = transform_previous_owners_imputer(X_test, state=owners_state)

# c) CATEGORICAL RESOLVERS - TEST (TRANSFORM ONLY)
X_test, _, _ = transform_ambiguous_brands(X_test, brand_state)
X_test, _, _ = transform_invalid_models(X_test, model_state)
X_test, _, _ = transform_transmission_resolver(X_test, transm_state)
X_test, _, _ = transform_fueltype_resolver(X_test, fuel_state)

# d) ENCODING - TEST (TRANSFORM ONLY)
X_test_high = te.transform(X_test[high_card_features])
X_test_low  = ohe.transform(X_test[low_card_features])
X_test_cat  = pd.concat([X_test_high, X_test_low], axis=1)

# e) SCALING - TEST (TRANSFORM ONLY)
X_test_num = scaler.transform(X_test[numeric_features])
X_test_num_df = pd.DataFrame(
    X_test_num,
    index=X_test.index,
    columns=[f"num_{col}" for col in numeric_features]
)

# 9) FINAL MATRIX & PREDICTION
X_test_final = pd.concat([X_test_num_df, X_test_cat], axis=1)

# Ensure same column order as training
X_test_final = X_test_final[X_full_final.columns]

print("X_test_final shape:", X_test_final.shape)

y_test_pred = ridge_final.predict(X_test_final)

print("Predictions summary (float):")
print(pd.Series(y_test_pred).describe())

# Optional rounding
y_test_pred_round = np.round(y_test_pred).astype(int)

submission = pd.DataFrame({
    "carID": test_df["carID"].astype(int),
    "price": y_test_pred  
})

sub_name = f"ridge_regression_final_submission_alpha{final_config_ridge['alpha']:.4f}.csv"
submission.to_csv(sub_name, index=False)
print(f"Submission file created: {sub_name}")

Final Ridge config used for train: {'alpha': np.float64(14.251026703029993)}
Preparing final Ridge Regression training...
Train full - Invalid Brands remaining: 0
Train full - Invalid Models remaining: 1
X_full_cat shape: (75973, 10)
X_full_num_df shape: (75973, 6)
X_full_final shape: (75973, 16)
Training final Ridge Regression model on full data...
Done.
X_test_final shape: (32567, 16)
Predictions summary (float):
count    32567.000000
mean     16897.809404
std       8754.244318
min     -19425.589300
25%      10847.683738
50%      15466.799959
75%      22289.102861
max      95024.131335
dtype: float64
Submission file created: ridge_regression_final_submission_alpha14.2510.csv
